# Incerteza da rede lida por dentro + regras fuzzy (versão enxuta)

Todas as features vêm do **comportamento interno da rede** (nada medido nos pixels), em três eixos:
se ela muda de ideia ao longo das camadas, se muda de ideia entre versões espelhadas da mesma imagem,
e o que ela respondeu para casos parecidos do treino.
Sistemas fuzzy aprendem sozinhos regras do tipo *SE trocas_opiniao é ALTA ... ENTÃO ERRO*.

Saída: `incerteza_fuzzy_resultados.xlsx`. **Mande esse arquivo** a cada rodada: a aba `diagnostico_features` é a que eu uso para dizer quais features melhorar.

In [ ]:
%%writefile leitura_rede.py
"""
leitura_rede.py — features do COMPORTAMENTO INTERNO da rede (nada medido nos pixels da imagem).

Três eixos de informação, escolhidos pela análise das rodadas 4 e 5:

1) A REDE MUDA DE IDEIA AO LONGO DAS CAMADAS?
   Em cada sub-bloco, um classificador simples (probe) diz o que a rede responderia se parasse ali.
   Os probes imitam a DECISÃO FINAL da rede, então a trajetória mostra quando ela se compromete.
     trocas_opiniao   quantas vezes a opinião troca entre maligno e benigno ao longo das camadas
     prof_decisao     a partir de que fração da rede a opinião não muda mais (0 = cedo, 1 = só no fim)
     entropia_probes  indecisão média das camadas
     conf_fim         o quanto as últimas camadas já concordam com a resposta final
     salto_final      a cabeça da rede discorda do último bloco

2) A REDE MUDA DE IDEIA ENTRE VERSÕES DA MESMA IMAGEM? (espelhada / invertida — a lesão é a mesma)
     vistas_mudam     fração das versões em que a decisão final muda
     p_vistas_std     quanto P varia entre as versões
     trajetoria_vistas quanto a trajetória de opinião nas camadas varia entre as versões

3) O QUE A REDE ACHA DE CASOS PARECIDOS DO TREINO? (vizinhos no espaço interno do último bloco)
     vizinhos_discordam  fração dos vizinhos para os quais a rede deu a resposta oposta
     dist_vizinhos       quão longe estão os vizinhos (1 = típico)

   + prox_p: proximidade de P ao limiar (1 = em cima do limiar). É a referência a superar.

Também devolve a trajetória crua (probe_b0..probe_bN) para análise.
"""

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image

EPS = 1e-9
FEATURES = ["trocas_opiniao", "prof_decisao", "entropia_probes", "conf_fim", "salto_final",
            "vistas_mudam", "p_vistas_std", "trajetoria_vistas",
            "vizinhos_discordam", "dist_vizinhos", "prox_p"]
VISTAS = {"original": lambda x: x, "espelho_h": lambda x: torch.flip(x, [3]),
          "espelho_v": lambda x: torch.flip(x, [2]), "giro_180": lambda x: torch.flip(x, [2, 3])}


def blocos_efficientnet(model, finos=True):
    """finos=True: os 18 sub-blocos do EfficientNet-B0; False: os 9 estágios."""
    if not finos:
        return list(model.features.children())
    out = [model.features[0]]
    for estagio in list(model.features.children())[1:-1]:
        out += list(estagio.children())
    return out + [model.features[-1]]


class LeitorRede:
    def __init__(self, model, limiar=0.5, blocos=None, k_vizinhos=15):
        self.model, self.limiar, self.k = model, limiar, k_vizinhos
        self.blocos = list(blocos) if blocos is not None else list(model.features.children())
        self.camada_final = [m for m in model.classifier.modules() if isinstance(m, nn.Linear)][-1]
        self.B = len(self.blocos)

    @torch.no_grad()
    def _ler(self, lotes, vista=lambda x: x):
        """Ativações médias por bloco e P, para cada imagem (aplicando a 'vista' antes)."""
        self.model.eval()
        acum, P = [[] for _ in range(self.B)], []
        for x in lotes:
            cap = [None] * self.B
            hs = [b.register_forward_hook(lambda m, i, o, k=k: cap.__setitem__(k, o.mean((2, 3)).cpu().numpy()))
                  for k, b in enumerate(self.blocos)]
            try:
                P.append(self.model(vista(x)).view(-1).cpu().numpy())
            finally:
                for h in hs: h.remove()
            for k in range(self.B): acum[k].append(cap[k])
        return [np.concatenate(a).astype(np.float64) for a in acum], np.concatenate(P)

    def fit(self, lotes, rotulos):
        """Referência = imagens que a rede viu no treino."""
        from sklearn.linear_model import LogisticRegression
        from sklearn.pipeline import make_pipeline
        from sklearn.preprocessing import StandardScaler
        A, P = self._ler(lotes)
        dec = (P >= self.limiar).astype(int)
        alvo = dec if len(np.unique(dec)) == 2 else np.asarray(rotulos).astype(int)
        self.probes = [make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=3000, class_weight="balanced"))
                       .fit(a, alvo) for a in A]
        self.banco = self._unit(A[-1]); self.banco_dec = dec                      # vizinhos: último bloco
        d = self._knn_dist(self.banco, excluir_si=True)
        self.dist_ref = float(np.median(d)) + EPS
        return self

    @staticmethod
    def _unit(a):
        a = a - a.mean(1, keepdims=True)
        return a / (np.linalg.norm(a, axis=1, keepdims=True) + EPS)

    def _knn(self, U, excluir_si=False):
        S = U @ self.banco.T
        if excluir_si:
            np.fill_diagonal(S, -np.inf)
        return np.argsort(-S, 1)[:, :self.k], S

    def _knn_dist(self, U, excluir_si=False):
        idx, S = self._knn(U, excluir_si)
        return (1 - np.take_along_axis(S, idx, 1)).mean(1)

    def _trajetoria(self, A):
        return np.stack([pr.predict_proba(a)[:, 1] for pr, a in zip(self.probes, A)], 1)   # (n, B)

    def transform(self, fazer_lotes) -> pd.DataFrame:
        """fazer_lotes: função sem argumentos que devolve um gerador novo de lotes (lido uma vez por vista)."""
        res = {nome: self._ler(fazer_lotes(), v) for nome, v in VISTAS.items()}
        A, P = res["original"]
        pp = self._trajetoria(A)
        final = (P >= self.limiar).astype(int)
        f = {"p": P}
        # 1) ao longo das camadas
        op = (pp >= 0.5).astype(int)
        f["trocas_opiniao"] = np.abs(np.diff(op, axis=1)).sum(1)
        f["prof_decisao"] = np.where(op != final[:, None], np.arange(self.B)[None, :] + 1, 0).max(1) / self.B
        f["entropia_probes"] = -(pp * np.log(pp + EPS) + (1 - pp) * np.log(1 - pp + EPS)).mean(1)
        conf = np.where(final[:, None] == 1, pp, 1 - pp)
        f["conf_fim"] = conf[:, -max(1, self.B // 3):].mean(1)
        f["salto_final"] = np.abs(P - pp[:, -1])
        # 2) entre versões da mesma imagem
        Ps = np.stack([res[v][1] for v in VISTAS], 1)                                         # (n, V)
        f["vistas_mudam"] = ((Ps[:, 1:] >= self.limiar).astype(int) != final[:, None]).mean(1)
        f["p_vistas_std"] = Ps.std(1)
        T = np.stack([self._trajetoria(res[v][0]) for v in VISTAS], 0)                        # (V, n, B)
        f["trajetoria_vistas"] = T.std(0).mean(1)
        # 3) vizinhos no espaço interno
        idx, S = self._knn(self._unit(A[-1]))
        f["vizinhos_discordam"] = (self.banco_dec[idx] != final[:, None]).mean(1)
        f["dist_vizinhos"] = (1 - np.take_along_axis(S, idx, 1)).mean(1) / self.dist_ref
        # referência
        f["prox_p"] = 1 - np.abs(P - self.limiar) / max(self.limiar, 1 - self.limiar)
        for k in range(self.B):
            f[f"probe_b{k}"] = pp[:, k]
        return pd.DataFrame(f)


def lotes_de_imagens(caminhos, tfm, device, bs=32):
    caminhos = list(caminhos)
    for i in range(0, len(caminhos), bs):
        yield torch.stack([tfm(Image.open(c).convert("RGB")) for c in caminhos[i:i + bs]]).to(device)


def ler(leitor, df, tfm, device, bs=32):
    """Uma linha por imagem: features + y_true, y_pred, erro."""
    t = leitor.transform(lambda: lotes_de_imagens(df.image_path, tfm, device, bs))
    t.insert(0, "image_id", df.image_id.values)
    t["y_true"] = df.is_malignant.values.astype(int)
    t["y_pred"] = (t.p >= leitor.limiar).astype(int)
    t["erro"] = (t.y_pred != t.y_true).astype(int)
    return t

In [ ]:
%%writefile regras_fuzzy.py
"""
regras_fuzzy.py — árvore de decisão fuzzy que aprende, sozinha, regras SE...ENTÃO ERRO/ACERTO,
e as ferramentas para avaliar e diagnosticar as features.
"""

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

EPS = 1e-12
TERMOS = ["BAIXA", "MÉDIA", "ALTA"]


# ----------------------------------------------------------------------------- partições
class Particoes:
    """3 termos triangulares por feature, centros nos quantis 5%, 50%, 95% (somam 1)."""

    def fit(self, X):
        C = np.nanquantile(np.asarray(X, float), [0.05, 0.5, 0.95], axis=0).T
        for j in range(len(C)):
            for k in (1, 2):
                C[j, k] = max(C[j, k], C[j, k - 1] + 1e-6 * (abs(C[j, k - 1]) + 1))
        self.centros = C
        return self

    def pertinencias(self, X):
        X = np.asarray(X, float); out = []
        for j, c in enumerate(self.centros):
            x = np.nan_to_num(X[:, j], nan=c[1])
            M = np.zeros((len(x), 3))
            M[:, 0] = x <= c[0]; M[:, 2] = x >= c[2]
            for k in (0, 1):
                d = (x >= c[k]) & (x < c[k + 1]); t = (x[d] - c[k]) / (c[k + 1] - c[k])
                M[d, k] = 1 - t; M[d, k + 1] = t
            out.append(M)
        return out


# ----------------------------------------------------------------------------- árvore fuzzy
class ArvoreFuzzy:
    """Fuzzy ID3: em cada nó, divide nos termos BAIXA/MÉDIA/ALTA da feature com maior ganho de informação."""

    def __init__(self, prof_max=2, min_peso=0.03, pureza=0.95, balancear=True):
        self.prof_max, self.min_peso, self.pureza, self.balancear = prof_max, min_peso, pureza, balancear

    def _H(self, w):
        b = np.array([(w * (self._y == c)).sum() * self._cw[c] for c in (0, 1)])
        if b.sum() <= EPS:
            return 0.0, np.array([.5, .5])
        p = b / b.sum()
        return float(-(p * np.log2(p + EPS)).sum()), p

    def fit(self, X, y, nomes):
        self.nomes = list(nomes); y = np.asarray(y).astype(int)
        self.part = Particoes().fit(X)
        self._M, self._y, self._n = self.part.pertinencias(X), y, len(y)
        self._cw = {c: (len(y) / (2 * max((y == c).sum(), 1)) if self.balancear else 1.0) for c in (0, 1)}
        self.raiz = self._crescer(np.ones(len(y)), (), 0, None)
        del self._M, self._y
        return self

    def _crescer(self, w, usados, prof, p_pai):
        H, p = self._H(w); peso = w.sum() / self._n
        if peso < self.min_peso:
            return {"folha": True, "p": p if p_pai is None else p_pai, "peso": peso}
        folha = {"folha": True, "p": p, "peso": peso}
        if prof >= self.prof_max or p.max() >= self.pureza:
            return folha
        melhor, ganho_max = None, 1e-3
        for j in range(len(self.nomes)):
            if j in usados: continue
            filhos = [w * self._M[j][:, k] for k in range(3)]
            tot = sum(f.sum() for f in filhos) + EPS
            g = H - sum(f.sum() / tot * self._H(f)[0] for f in filhos)
            if g > ganho_max: melhor, ganho_max = (j, filhos), g
        if melhor is None:
            return folha
        j, filhos = melhor
        return {"folha": False, "feat": j, "peso": peso, "p": p,
                "filhos": [self._crescer(f, usados + (j,), prof + 1, p) for f in filhos]}

    def predict_proba(self, X):
        M = self.part.pertinencias(X); n = len(M[0]); num = np.zeros(n); den = np.zeros(n)

        def desce(no, g):
            if no["folha"]:
                num[:] += g * no["p"][1]; den[:] += g; return
            for k, filho in enumerate(no["filhos"]):
                gk = g * M[no["feat"]][:, k]
                if gk.max() > 0: desce(filho, gk)
        desce(self.raiz, np.ones(n))
        p1 = np.where(den > EPS, num / np.maximum(den, EPS), .5)
        return np.c_[1 - p1, p1]

    def _folhas(self):
        out = []
        def rec(no, cam):
            if no["folha"]: out.append((cam, no)); return
            for k, f in enumerate(no["filhos"]): rec(f, cam + [(no["feat"], k)])
        rec(self.raiz, []); return out

    @property
    def n_regras(self):
        return len(self._folhas())

    def assinaturas(self):
        return {(tuple(sorted(c)), int(np.argmax(no["p"]))) for c, no in self._folhas()}

    def regras(self):
        lin = []
        for cam, no in sorted(self._folhas(), key=lambda t: -t[1]["peso"]):
            se = " E ".join(f"{self.nomes[j]} é {TERMOS[k]}" for j, k in cam) or "(sempre)"
            c = int(np.argmax(no["p"]))
            lin.append({"regra": f"SE {se} ENTÃO {'ERRO' if c else 'ACERTO'}",
                        "confiança": round(float(no["p"][c]), 3), "suporte": round(float(no["peso"]), 4)})
        return pd.DataFrame(lin)


# ----------------------------------------------------------------------------- fuzzy aditivo
class FuzzyAditivo:
    """Regras de UMA condição cada, somadas: 'SE salto_final é ALTA → +1,28 no risco de erro'.
    Os pesos (em log-odds) são aprendidos por regressão logística sobre os graus de pertinência.
    Peso positivo = aumenta o risco de a rede errar; negativo = diminui."""

    def __init__(self, C=1.0):
        self.C = C

    def fit(self, X, y, nomes):
        self.nomes = list(nomes); self.part = Particoes().fit(X)
        self.lr = LogisticRegression(C=self.C, max_iter=3000).fit(np.hstack(self.part.pertinencias(X)), y)
        return self

    def predict_proba(self, X):
        return self.lr.predict_proba(np.hstack(self.part.pertinencias(X)))

    @property
    def n_regras(self):
        return 3 * len(self.nomes)

    def regras(self):
        w = self.lr.coef_[0]
        lin = [{"regra": f"SE {n} é {TERMOS[k]}", "peso no risco de erro": round(float(w[3 * j + k]), 3),
                "efeito": "aumenta" if w[3 * j + k] > 0.05 else ("diminui" if w[3 * j + k] < -0.05 else "neutro")}
               for j, n in enumerate(self.nomes) for k in range(3)]
        lin.append({"regra": "(base)", "peso no risco de erro": round(float(self.lr.intercept_[0]), 3), "efeito": ""})
        return pd.DataFrame(lin)



# ----------------------------------------------------------------------------- Wang-Mendel
class WangMendel:
    """Wang & Mendel (1992): cada exemplo propõe a regra com o termo de maior grau em cada feature;
    a conclusão (ERRO/ACERTO) e o fator de certeza (CF) vêm de todos os exemplos compatíveis, com classes balanceadas.
    Na inferência, todas as regras votam, pesadas por compatibilidade × CF."""

    def __init__(self, min_suporte=0.01, min_cf=0.1):
        self.min_suporte, self.min_cf = min_suporte, min_cf

    def fit(self, X, y, nomes):
        y = np.asarray(y).astype(int); self.nomes = list(nomes); self.part = Particoes().fit(X)
        M = self.part.pertinencias(X); n = len(y)
        w = np.where(y == 1, n / (2 * max(y.sum(), 1)), n / (2 * max((1 - y).sum(), 1)))
        antecedentes = {tuple(int(np.argmax(M[j][i])) for j in range(len(M))) for i in range(n)}
        self.regras_ = []
        for ant in antecedentes:
            comp = np.prod([M[j][:, k] for j, k in enumerate(ant)], axis=0)
            sup = comp.sum() / n
            b = np.array([(comp * w)[y == 0].sum(), (comp * w)[y == 1].sum()])
            if sup < self.min_suporte or b.sum() <= EPS: continue
            c = int(b.argmax()); cf = (b[c] - b[1 - c]) / b.sum()
            if cf >= self.min_cf:
                self.regras_.append({"ant": ant, "cons": c, "cf": float(cf), "suporte": float(sup)})
        self.regras_.sort(key=lambda r: -r["suporte"] * r["cf"])
        return self

    def predict_proba(self, X):
        M = self.part.pertinencias(X); S = np.zeros((len(M[0]), 2))
        for r in self.regras_:
            S[:, r["cons"]] += np.prod([M[j][:, k] for j, k in enumerate(r["ant"])], axis=0) * r["cf"]
        tot = S.sum(1, keepdims=True)
        return np.where(tot > EPS, S / np.maximum(tot, EPS), 0.5)

    @property
    def n_regras(self):
        return len(self.regras_)

    def assinaturas(self):
        return {(r["ant"], r["cons"]) for r in self.regras_}

    def regras(self):
        return pd.DataFrame([{"regra": "SE " + " E ".join(f"{self.nomes[j]} é {TERMOS[k]}" for j, k in enumerate(r["ant"]))
                              + (" ENTÃO ERRO" if r["cons"] else " ENTÃO ACERTO"),
                              "certeza (CF)": round(r["cf"], 3), "suporte": round(r["suporte"], 3)} for r in self.regras_])

# ----------------------------------------------------------------------------- seleção e avaliação
def selecionar(tab, cand, k=4, max_corr=0.85):
    """Top-k por AUROC (para detectar o erro), sem features muito correlacionadas entre si."""
    rk = sorted(cand, key=lambda f: -abs(roc_auc_score(tab.erro, tab[f]) - 0.5) if tab[f].nunique() > 1 else 0)
    esc = []
    for f in rk:
        if all(abs(spearmanr(tab[f], tab[g]).correlation) < max_corr for g in esc):
            esc.append(f)
        if len(esc) == k: break
    return esc


def comparar(tab, cand, k=4, prof=2, n_folds=5, seed=42):
    """Validação cruzada: só P × regressão logística × árvore fuzzy. Seleção de features dentro de cada fold."""
    y = tab.erro.values
    oof = {m: np.zeros(len(y)) for m in ("Só P", "Regressão logística", "Árvore fuzzy", "Fuzzy aditivo")}
    escolhas, n_regras = [], []
    for tr, te in StratifiedKFold(n_folds, shuffle=True, random_state=seed).split(tab, y):
        d = tab.iloc[tr]; feats = selecionar(d, cand, k); escolhas.append(feats)
        oof["Só P"][te] = tab.prox_p.values[te]
        lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)).fit(d[feats], y[tr])
        oof["Regressão logística"][te] = lr.predict_proba(tab.iloc[te][feats])[:, 1]
        arv = ArvoreFuzzy(prof).fit(d[feats].values, y[tr], feats)
        oof["Árvore fuzzy"][te] = arv.predict_proba(tab.iloc[te][feats].values)[:, 1]; n_regras.append(arv.n_regras)
        oof["Fuzzy aditivo"][te] = FuzzyAditivo().fit(d[feats].values, y[tr], feats).predict_proba(tab.iloc[te][feats].values)[:, 1]
    r = np.random.default_rng(0); linhas = []
    for m, s in oof.items():
        d = [roc_auc_score(y[i], s[i]) - roc_auc_score(y[i], oof["Só P"][i])
             for i in (r.integers(0, len(y), len(y)) for _ in range(1000)) if y[i].min() != y[i].max()]
        o = np.argsort(s); manter = o[: int(round(.8 * len(y)))]
        linhas.append({"método": m, "AUROC": roc_auc_score(y, s), "AUPRC": average_precision_score(y, s),
                       "ganho sobre só P": roc_auc_score(y, s) - roc_auc_score(y, oof["Só P"]),
                       "IC 95% do ganho": "—" if m == "Só P" else f"[{np.percentile(d, 2.5):+.3f}, {np.percentile(d, 97.5):+.3f}]",
                       "erro com 20% encaminhados": y[manter].mean(), "erro sem encaminhar": y.mean(),
                       "nº regras": np.mean(n_regras) if m == "Árvore fuzzy" else (3 * k if m == "Fuzzy aditivo" else np.nan)})
    return pd.DataFrame(linhas), escolhas, oof


def estabilidade(tab, feats, prof=2, n=10, seed=42):
    """Jaccard médio das regras entre reamostragens bootstrap (1 = sempre as mesmas)."""
    r = np.random.default_rng(seed); A = []
    for _ in range(n):
        i = r.integers(0, len(tab), len(tab))
        A.append(ArvoreFuzzy(prof).fit(tab.iloc[i][feats].values, tab.erro.values[i], feats).assinaturas())
    return float(np.mean([len(a & b) / max(len(a | b), 1) for j, a in enumerate(A) for b in A[j + 1:]]))


def diagnostico(tab, cand, escolhas=None, faixas=5):
    """Uma linha por feature, para decidir o que melhorar:
       AUROC           o quanto ela sozinha separa erros de acertos (0,5 = nada)
       AUROC além de P o mesmo, mas dentro de faixas de P parecido: o que ela traz de NOVO em relação a P
       corr. com P     redundância com P (perto de ±1 = só repete P)
       escolhida       em quantos folds a seleção a escolheu
       mediana acerto / erro, nº de valores distintos (poucos = feature grosseira, pouca informação)
       mais parecida com: a outra feature mais correlacionada (perto de 1 = redundante)"""
    y = tab.erro.values
    bins = pd.qcut(tab.prox_p.rank(method="first"), faixas, labels=False)
    lin = []
    for f in cand:
        v = tab[f].values
        if np.nanstd(v) == 0:
            lin.append({"feature": f, "AUROC": .5, "AUROC além de P": .5, "corr. com P": 0}); continue
        a = roc_auc_score(y, v); dentro = []
        for b in range(faixas):
            m = bins.values == b
            if y[m].min() != y[m].max() and np.std(v[m]) > 0:
                ab = roc_auc_score(y[m], v[m]); dentro.append((max(ab, 1 - ab), m.sum()))
        lin.append({"feature": f, "AUROC": max(a, 1 - a), "direção": "maior = erro" if a >= .5 else "menor = erro",
                    "AUROC além de P": np.average([d for d, _ in dentro], weights=[w for _, w in dentro]) if dentro else np.nan,
                    "corr. com P": spearmanr(v, tab.prox_p).correlation,
                    "escolhida (folds)": sum(f in e for e in escolhas) if escolhas else np.nan,
                    "mediana acertos": np.median(v[y == 0]), "mediana erros": np.median(v[y == 1]),
                    "nº valores distintos": len(np.unique(v))})
    out = pd.DataFrame(lin)
    R = tab[cand].corr("spearman").abs().fillna(0)
    R = R.mask(np.eye(len(R), dtype=bool), 0)
    out["mais parecida com"] = [f"{R[f].idxmax()} ({R[f].max():.2f})" if f in R else "" for f in out.feature]
    return out.sort_values("AUROC além de P", ascending=False).reset_index(drop=True)

## 1. Configuração

In [ ]:
METADATA_CSV = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv'
IMG_DIRS = ['/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1',
            '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_2']
MODEL_PT    = '/kaggle/input/models/matheusbuttow/modelo-bracis/pytorch/default/1/effnet_ham_best.pt'
SPLIT_CSV   = None   # do notebook de retreino (recomendado); None = split antigo do BRACIS
LIMIAR_JSON = None   # do notebook de retreino; None = usa LIMIAR
LIMIAR      = 0.5
N_REF       = 1000   # imagens do treino usadas como referência
K_FEATURES  = 4      # features por árvore
PROF        = 2      # profundidade da árvore fuzzy (2 = regras curtas e estáveis)
SUBBLOCOS   = True   # lê os 18 sub-blocos do EfficientNet (trajetória mais detalhada) em vez dos 9 estágios
SEED, BS    = 42, 32

## 2. Dados e modelo

In [ ]:
import os, json, numpy as np, pandas as pd, torch, torch.nn as nn, warnings
from pathlib import Path
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
import leitura_rede as lr, regras_fuzzy as rf
warnings.filterwarnings('ignore'); pd.set_option('display.width', 220); pd.set_option('display.max_columns', 20)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'; torch.manual_seed(SEED)

meta = pd.read_csv(METADATA_CSV)
meta['is_malignant'] = meta.dx.isin({'mel', 'bcc', 'akiec'}).astype(int)
caminho = {a.stem: str(a) for d in IMG_DIRS for a in Path(d).glob('*.jpg')}
meta['image_path'] = meta.image_id.map(caminho); meta = meta.dropna(subset=['image_path'])
if LIMIAR_JSON: LIMIAR = json.load(open(LIMIAR_JSON))['limiar']
if SPLIT_CSV:
    meta = meta.merge(pd.read_csv(SPLIT_CSV)[['image_id', 'split']], on='image_id')
    treino, cal = meta[meta.split == 'train'], meta[meta.split == 'test']
else:
    m3 = meta.iloc[:3000]
    treino, cal = train_test_split(m3, test_size=0.2, stratify=m3.is_malignant, random_state=42)
    cal = cal[~cal.lesion_id.isin(treino.lesion_id)]          # tira lesões repetidas
ref = treino.sample(n=min(N_REF, len(treino)), random_state=SEED)
cal = cal.reset_index(drop=True)
print(f'referência: {len(ref)} | calibração: {len(cal)} imagens | limiar: {LIMIAR:.3f}')

model = models.efficientnet_b0(weights=None)
model.classifier = nn.Sequential(nn.Dropout(0.4), nn.Linear(1280, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, 1), nn.Sigmoid())
model.load_state_dict(torch.load(MODEL_PT, map_location=DEVICE)); model.to(DEVICE).eval()
tfm = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(),
                          transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

## 3. Ler a rede por dentro

In [ ]:
leitor = lr.LeitorRede(model, limiar=LIMIAR, blocos=lr.blocos_efficientnet(model, SUBBLOCOS)).fit(lr.lotes_de_imagens(ref.image_path, tfm, DEVICE, BS), ref.is_malignant)
tab = lr.ler(leitor, cal, tfm, DEVICE, BS)
print(f'Erros da rede: {tab.erro.sum()} de {len(tab)} ({tab.erro.mean():.1%}) | a rede lê cada imagem em 4 versões (original e espelhadas)')
tab[['p', 'erro'] + lr.FEATURES].head()

### Como a opinião da rede evolui ao longo das camadas

Para cada camada: chance de a rede, se parasse ali, já dar a resposta final. Linha = mediana; faixa = metade central das imagens.
Nos erros, espera-se uma curva mais baixa e irregular (a rede hesita e muda de ideia).

In [ ]:
import matplotlib.pyplot as plt
B = leitor.B; PB = tab[[f'probe_b{k}' for k in range(B)]].values
conf = np.where(tab.y_pred.values[:, None] == 1, PB, 1 - PB)
fig, ax = plt.subplots(figsize=(7, 3.6))
for nome, m, cor in [('acertos', tab.erro == 0, '#2a78d6'), ('erros', tab.erro == 1, '#eb6834')]:
    q = np.percentile(conf[m.values], [25, 50, 75], axis=0)
    ax.plot(range(1, B + 1), q[1], color=cor, lw=2, label=f'{nome} (n={m.sum()})')
    ax.fill_between(range(1, B + 1), q[0], q[2], color=cor, alpha=0.15)
ax.axhline(0.5, color='gray', ls='--', lw=1); ax.set_xlabel('bloco da rede'); ax.set_ylabel('concorda com a resposta final')
ax.set_ylim(0, 1); ax.legend(frameon=False); ax.spines[['top', 'right']].set_visible(False); plt.tight_layout(); plt.show()

### Exemplos com as fotos reais

Três lesões do seu conjunto: o acerto mais tranquilo, o erro com mais mudanças de ideia e o erro mais confiante (P longe do limiar).
Ao lado de cada foto, a curva de opinião da rede ao longo dos sub-blocos (abaixo de 0,5 = aquela camada pensava o contrário).

In [ ]:
from PIL import Image
ex = [('Acerto tranquilo', tab[tab.erro == 0].entropia_probes.idxmin()),
      ('Erro hesitante', tab[tab.erro == 1].trocas_opiniao.idxmax()),
      ('Erro confiante', tab[tab.erro == 1].prox_p.idxmin())]
fig, axes = plt.subplots(len(ex), 2, figsize=(9, 3.2 * len(ex)), gridspec_kw={'width_ratios': [1, 1.6]})
for (nome, i), (a1, a2) in zip(ex, axes):
    r = tab.loc[i]
    a1.imshow(Image.open(cal.image_path.iloc[i]).convert('RGB')); a1.axis('off')
    a1.set_title(f"{nome}\n{r.image_id} | real: {'maligna' if r.y_true else 'benigna'} | rede: {'maligna' if r.y_pred else 'benigna'} (P={r.p:.2f})", fontsize=9, loc='left')
    a2.plot(range(1, B + 1), conf[i], marker='o', color='#b8325a' if r.erro else '#2a78d6')
    a2.axhline(0.5, color='gray', ls='--', lw=1); a2.set_ylim(0, 1); a2.set_xlabel('sub-bloco'); a2.set_ylabel('concorda com a resposta final')
    a2.set_title(f"mudou de ideia {int(r.trocas_opiniao)}x | decidiu a partir de {r.prof_decisao:.0%} da rede | firmeza no fim {r.conf_fim:.2f}", fontsize=9, loc='left')
    a2.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()

## 4. Diagnóstico das features

Uma linha por feature. A coluna mais importante é **AUROC além de P**: o quanto a feature separa erros de acertos
entre imagens com P parecido, ou seja, o que ela traz de novo. Compare com a linha de `prox_p` (o próprio P).

In [ ]:
res, escolhas, oof = rf.comparar(tab, lr.FEATURES, k=K_FEATURES, prof=PROF, seed=SEED)
diag = rf.diagnostico(tab, lr.FEATURES, escolhas)
diag.round(3)

## 5. Regras fuzzy × só P

Dois sistemas fuzzy, ambos com regras aprendidas sozinhas:
- **Árvore fuzzy**: regras com 2 condições (*SE A é ALTA E B é BAIXA ENTÃO ERRO*).
- **Fuzzy aditivo**: regras de 1 condição, cada uma com um peso, somadas (*SE salto_final é ALTA → +1,3 no risco*).
- **Wang-Mendel**: cada imagem propõe uma regra com 3 condições; as regras mais estáveis entre os três (com 3 features).
  Na rodada 4 ele chegou mais perto da regressão logística (0,83 contra 0,81 da árvore).

In [ ]:
display(res.round(3))
feats = rf.selecionar(tab, lr.FEATURES, K_FEATURES)
arv = rf.ArvoreFuzzy(PROF).fit(tab[feats].values, tab.erro.values, feats)
feats_wm = rf.selecionar(tab, lr.FEATURES, 3)                       # Wang-Mendel: 3 features
wm = rf.WangMendel().fit(tab[feats_wm].values, tab.erro.values, feats_wm)
adi = rf.FuzzyAditivo().fit(tab[feats].values, tab.erro.values, feats)
regras, regras_adi = arv.regras(), adi.regras()
est = rf.estabilidade(tab, feats, PROF)
print(f'Features: {feats} | estabilidade das regras da árvore: {est:.2f} (1 = sempre as mesmas)')
display(regras); display(regras_adi)
print(f'Wang-Mendel ({wm.n_regras} regras) com {feats_wm}:'); display(wm.regras())

## 6. Salvar (mande este arquivo)

In [ ]:
resumo = pd.Series({'imagens de calibração': len(tab), 'erros da rede': int(tab.erro.sum()), 'limiar': LIMIAR,
                    'split': 'por lesão (retreino)' if SPLIT_CSV else 'antigo (BRACIS)', 'features da árvore': ', '.join(feats),
                    'estabilidade das regras': round(est, 3), 'blocos lidos': leitor.B, 'sub-blocos': SUBBLOCOS, 'imagens de referência': len(ref)})
with pd.ExcelWriter('incerteza_fuzzy_resultados.xlsx') as xw:
    resumo.to_frame('valor').to_excel(xw, sheet_name='resumo')
    diag.round(4).to_excel(xw, sheet_name='diagnostico_features', index=False)
    res.round(4).to_excel(xw, sheet_name='comparacao', index=False)
    regras.to_excel(xw, sheet_name='regras', index=False)
    regras_adi.to_excel(xw, sheet_name='regras_aditivo', index=False)
    wm.regras().to_excel(xw, sheet_name='regras_wang_mendel', index=False)
    tab.to_excel(xw, sheet_name='features_por_imagem', index=False)
print('Salvo: incerteza_fuzzy_resultados.xlsx')